In [ ]:
#Importing dependencies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
#Opening .csv file 

df = pd.read_csv("C:\Users\User\Columbia-DBMI-\Acccelometer_Gyrometer\preprocessed\preprocessed_typing.csv")
df

In [ ]:
#Checking nulls and label
print(df["condition"].value_counts())

print(df.isnull().sum())

In [ ]:
#Dropping min and max because they might not be preidctive of the modelm
for col in df.columns:
    if col.endswith("max") or col.endswith("min"):
        df = df.drop(columns = [col])

df

In [ ]:
#Balancing the class label with smote

#Defining features and labels 
y = df["condition"]
X = df.drop(columns = ["condition", "patient_id", "wrist_LeftWrist", "wrist_RightWrist"])
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Original class distribution:", Counter(y))

smote= SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("Resampled class distribution:", Counter(y_resampled))
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size = 0.2, random_state = 42)


In [ ]:
#Training a RF with no hyperparamter
rfc = RandomForestClassifier()

#Fitting and training
rfc.fit(X_train, y_train)
pred = rfc.predict(X_test)

In [ ]:
#Eval metrics
print(accuracy_score(y_test, pred))

print(confusion_matrix(y_test, pred))\

print(classification_report(y_test, pred))

In [ ]:
#Tuning hyperparamtyers with GridSearCV
from sklearn.model_selection import GridSearchCV, KFold 

k = KFold(n_splits = 5)

param_rfc = {"n_estimators": [50, 100], "max_depth": [4, 8, 12], "min_samples_split": [4, 6], "min_samples_leaf": [2, 4, 6]}

grid_rfc = GridSearchCV(estimator = rfc, cv = k, param_grid=param_rfc)

In [ ]:
#Fitting and training the model
grid_rfc.fit(X_train, y_train)

In [ ]:
best_params = grid_rfc.best_params_

print(grid_rfc.best_score_)

In [ ]:
#Retrainig model based on performance
rfc_model = RandomForestClassifier(n_estimators = best_params["n_estimators"], max_depth = best_params["max_depth"], min_samples_split = best_params["min_samples_split"], min_samples_leaf = best_params["min_samples_leaf"])

In [ ]:
#Fitting and training 
rfc_model.fit(X_train, y_train)
pred_rfc = rfc_model.predict(X_test)

In [ ]:
#Eval metrics
print(accuracy_score(y_test, pred_rfc))

print(confusion_matrix(y_test, pred_rfc))

print(classification_report(y_test, pred_rfc))

In [ ]:
#Proceeding with feature selection
model_features = rfc_model.feature_importances_

#Creating a new df with the import features and training on the model
pd_features = pd.DataFrame({"Features": X_train.columns, "Feature_importance": model_features}).sort_values(by = "Feature_importance", ascending = False)
pd_features

In [ ]:
#Visualizations

plt.figure(figsize=(10, 6))
sns.barplot(
    data=pd_features,
    x="Feature_importance",
    y="Features",
    palette="viridis"
)
plt.title("Visualization of plot")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()